In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# import numpy as np

In [ ]:
# Categorical features
CAT_FEATURES = [
    "education_level",
    "has_partner",
    "had_partner",
    "is_female",
    "ever_smoker",
    "is_current_smoker",
    "drinking_frequency",
]

EXCLUDED_FEATURES = ["survey_weight"]

TARGET = "has_diabetes_or_prediabetes"

In [ ]:
df = pd.read_csv("../dataset/processed_data_combined_25features.csv")

In [ ]:
df.info()

In [ ]:
# Set style for better-looking plots
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)

In [ ]:
# Get numerical columns (excluding categorical, target, and excluded features)
all_excluded = CAT_FEATURES + EXCLUDED_FEATURES + [TARGET]
numerical_cols = [col for col in df.columns if col not in all_excluded]

In [ ]:
df[EXCLUDED_FEATURES].boxplot()

---

In [ ]:
PA_FEATURES = ["moderate_minutes_per_week", 
               "vigorous_minutes_per_week", 
               "sedentary_minutes_per_day"
]

SLEEP_FEATURES = ["sleep_hours_weekday", "sleep_hours_weekend"]

In [ ]:
df[PA_FEATURES].describe()

In [ ]:
df[SLEEP_FEATURES].describe()

---

## Distribution Comparison for Numerical Features

In [ ]:
# Define the sample weight column
SAMPLE_WEIGHT = "survey_weight"

# Get data for each target level
target_levels = df[TARGET].unique()

# Calculate number of rows needed (1 feature per row, 2 plots per feature)
n_features = len(numerical_cols)

# Create a single large figure with all numerical features
fig, axes = plt.subplots(n_features, 2, figsize=(16, 5 * n_features))

# Handle case where there's only one feature (axes won't be 2D)
if n_features == 1:
    axes = axes.reshape(1, -1)

for idx, col in enumerate(numerical_cols):
    # Left plot: Unweighted
    ax_unweighted = axes[idx, 0]
    for level in target_levels:
        data = df[df[TARGET] == level][col].dropna()
        ax_unweighted.hist(data, bins=30, alpha=0.5, label=f"{TARGET}={level}",
                          edgecolor="black", density=True)

    ax_unweighted.set_xlabel(col, fontsize=11)
    ax_unweighted.set_ylabel("Relative Frequency", fontsize=11)
    ax_unweighted.set_title(f"Unweighted: {col}", fontsize=12)
    ax_unweighted.legend(fontsize=10)

    # Right plot: Weighted
    ax_weighted = axes[idx, 1]
    for level in target_levels:
        mask = df[TARGET] == level
        data = df[mask][col].dropna()
        weights = df[mask].loc[data.index, SAMPLE_WEIGHT]

        ax_weighted.hist(data, bins=30, alpha=0.5, label=f"{TARGET}={level}",
                        edgecolor="black", density=True, weights=weights)

    ax_weighted.set_xlabel(col, fontsize=11)
    ax_weighted.set_ylabel("Weighted Relative Frequency", fontsize=11)
    ax_weighted.set_title(f"Weighted: {col}", fontsize=12)
    ax_weighted.legend(fontsize=10)

fig.suptitle(f"Distribution Comparison: Unweighted vs Weighted by {TARGET}",
             fontsize=16, y=0.998)
plt.tight_layout()
plt.show()

## Cross Tables for Categorical Features

In [ ]:
# Create visualizations for categorical features
n_cat_features = len(CAT_FEATURES)

# Create a single large figure with all categorical features
fig, axes = plt.subplots(n_cat_features, 2, figsize=(16, 5 * n_cat_features))

# Handle case where there's only one feature (axes won't be 2D)
if n_cat_features == 1:
    axes = axes.reshape(1, -1)

for idx, cat_col in enumerate(CAT_FEATURES):
    # Left plot: Stacked bar chart showing composition within each target level
    cross_tab = pd.crosstab(df[cat_col], df[TARGET])
    cross_tab_pct = cross_tab.div(cross_tab.sum(axis=0), axis=1) * 100

    cross_tab_pct.T.plot(kind="bar", stacked=True, ax=axes[idx, 0],
                         colormap="tab10", alpha=0.8, edgecolor="black")
    axes[idx, 0].set_xlabel(TARGET, fontsize=11)
    axes[idx, 0].set_ylabel("Percentage (%)", fontsize=11)
    axes[idx, 0].set_title(f"Composition of {cat_col} within each {TARGET} level", fontsize=12)
    axes[idx, 0].legend(title=cat_col, bbox_to_anchor=(1.05, 1), loc="upper left", fontsize=9)
    axes[idx, 0].set_xticklabels(axes[idx, 0].get_xticklabels(), rotation=0)
    axes[idx, 0].set_ylim(0, 100)

    # Right plot: Grouped bar chart showing target distribution within each feature level
    cross_tab_row_pct = cross_tab.div(cross_tab.sum(axis=1), axis=0) * 100

    cross_tab_row_pct.plot(kind="bar", ax=axes[idx, 1],
                           colormap="Set2", alpha=0.8, edgecolor="black", width=0.8)
    axes[idx, 1].set_xlabel(cat_col, fontsize=11)
    axes[idx, 1].set_ylabel("Percentage (%)", fontsize=11)
    axes[idx, 1].set_title(f"{TARGET} distribution within each {cat_col} level", fontsize=12)
    axes[idx, 1].legend(title=TARGET, fontsize=9)
    axes[idx, 1].set_xticklabels(axes[idx, 1].get_xticklabels(), rotation=45, ha="right")
    axes[idx, 1].set_ylim(0, 100)
    axes[idx, 1].axhline(y=50, color="gray", linestyle="--", linewidth=1, alpha=0.5)

fig.suptitle(f"Categorical Features vs {TARGET}", fontsize=16, y=0.998)
plt.tight_layout()
plt.show()

In [ ]:
df.loc[df["has_diabetes_or_prediabetes"] == 1]["vigorous_every_X_days"].value_counts()

In [ ]:
df.loc[df["has_diabetes_or_prediabetes"] == 0]["vigorous_every_X_days"].value_counts()

In [ ]:
df["has_diabetes_or_prediabetes"].value_counts()